# МТС · Qwen3-0.6B · step648 + 2000 · весь запуск≤5часов

Новый эксперимент на официальном пакете2.1. GPU ещё не запускалась. Ячейки1–4 по порядку. Нужен один payload ZIP.

Начальный/финальный dev32 входят в бюджет; полные383 и квантизация сюда не включены. Пять часов — наш верхний лимит, доступ Google может закончиться раньше. Таймер начинает считать с первой кодовой ячейки, включая ожидание загрузки. Все результаты сохраняются в Drive.

## 1. Время и сохранение

Выберите GPU в настройках Colab. Если уже работали сегодня, уменьшите AVAILABLE_MINUTES. Разрешите доступ к своему Drive. Сохраните напечатанный путь RUN.

In [ ]:
import time
_first_cell_started=time.time()  # До Drive, установки, загрузки и ожидания.
AVAILABLE_MINUTES=300  # Уменьшите, если часть сегодняшнего бюджета уже использована.
RUN_TRAINING=True
AUTO_DISCONNECT=True
EXISTING_RUN=""  # Для восстановления: полный путь ранее напечатанного RUN на Drive.
RESUME_NEW_CHECKPOINT=""  # Только собственный checkpoint нового запуска; не старый step648.
exec('"""Colab orchestration. One persisted wall-clock budget; results survive kernel reset."""\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nimport hashlib,json,os,subprocess,sys,threading,time,zipfile\n\nLOCATOR=Path(\'/content/mts_qwen06_v21_current.json\')\n\ndef atomic_json(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+\'.pending\');tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2)+\'\\n\',encoding=\'utf-8\');os.replace(tmp,path)\n\ndef arm_disconnect(deadline):\n    """VM release is best effort; process supervisor independently enforces its own cutoff."""\n    prior=globals().get(\'_release_timer\')\n    if prior:prior.cancel()\n    def release():\n        try:\n            from google.colab import runtime\n            runtime.unassign()\n        except Exception as exc:print(\'Не удалось отключить ВМ автоматически:\',repr(exc),flush=True)\n    timer=threading.Timer(max(0,deadline-time.time()),release);timer.daemon=True;timer.start()\n    globals()[\'_release_timer\']=timer\n    return timer\n\ndef setup_session(origin,available_minutes=300,existing_run=\'\',*,mount=True,arm=True):\n    if not 60<=available_minutes<=300:raise ValueError(\'AVAILABLE_MINUTES: от60 до300, с учётом уже использованного сегодня времени\')\n    if arm:arm_disconnect(origin+available_minutes*60)\n    if mount:\n        from google.colab import drive\n        drive.mount(\'/content/drive\')\n    if existing_run:\n        run=Path(existing_run)\n        if not (run/\'budget.json\').is_file():raise ValueError(\'EXISTING_RUN не содержит budget.json; не создаём пустую замену\')\n    elif LOCATOR.exists():\n        run=Path(json.loads(LOCATOR.read_text())[\'run\'])\n        if not (run/\'budget.json\').is_file():raise ValueError(\'Предыдущая папка недоступна; проверьте Drive/EXISTING_RUN\')\n    else:\n        run=Path(\'/content/drive/MyDrive/MTS_Qwen06_V21_runs\')/datetime.fromtimestamp(origin,timezone.utc).strftime(\'v21_%Y%m%dT%H%M%SZ\')\n        run.mkdir(parents=True,exist_ok=False)\n        atomic_json(run/\'budget.json\',{\'origin_epoch\':origin,\'total_minutes\':available_minutes,\'scope\':\'Entire experiment, not a guaranteed Google quota\'})\n    budget=json.loads((run/\'budget.json\').read_text());deadline=budget[\'origin_epoch\']+budget[\'total_minutes\']*60\n    if available_minutes!=budget[\'total_minutes\']:raise ValueError(\'Повторный запуск не меняет бюджет; сохранённый размер: \'+str(budget[\'total_minutes\']))\n    atomic_json(LOCATOR,{\'run\':str(run),\'root\':\'/content/mts_qwen06_v21_2000\'})\n    if arm:arm_disconnect(deadline)\n    print(\'Папка RUN:\',run,\'\\nОсталось общего времени, минут:\',round(max(0,deadline-time.time())/60,1),flush=True)\n    print(\'Бюджет включает подготовку, загрузку, обучение, оценку и архив. Доступность GPU Google не гарантируется.\')\n    return run,budget\n\ndef recover_run(existing_run=\'\'):\n    if existing_run:run=Path(existing_run)\n    elif LOCATOR.exists():run=Path(json.loads(LOCATOR.read_text())[\'run\'])\n    else:\n        candidates=sorted(Path(\'/content/drive/MyDrive/MTS_Qwen06_V21_runs\').glob(\'v21_*\'))\n        raise ValueError(\'Укажите EXISTING_RUN явно, чтобы не выбрать чужой запуск: \'+str([str(p) for p in candidates]))\n    if not (run/\'budget.json\').is_file():raise ValueError(\'Нет сохранённого budget.json: \'+str(run))\n    return run\n\ndef supervise(command,run,budget):\n    from ml.qwen.budget_v21 import BudgetStop\n    budget.require(15*60)\n    if (run/\'experiment_report.json\').is_file():\n        report=json.loads((run/\'experiment_report.json\').read_text())\n        if report.get(\'status\')==\'completed\' or \'--resume-checkpoint\' not in command:\n            raise ValueError(\'В этой папке уже есть отчёт. Используйте просмотр результатов; незавершённый запуск возобновляется явно из своего чекпоинта.\')\n    env=dict(os.environ,TOKENIZERS_PARALLELISM=\'false\',HF_HUB_DISABLE_PROGRESS_BARS=\'1\',PYTHONUNBUFFERED=\'1\')\n    with (run/\'console.log\').open(\'a\',encoding=\'utf-8\',buffering=1) as log:\n        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding=\'utf-8\',errors=\'replace\',bufsize=1,env=env)\n        def reader():\n            for line in process.stdout:\n                log.write(line);print(line,end=\'\',flush=True)\n        thread=threading.Thread(target=reader,daemon=True);thread.start()\n        interrupted=False\n        try:\n            while process.poll() is None:\n                if budget.left(\'supervisor\')<=0:raise BudgetStop(\'Пора сохранить результаты и освободить GPU\')\n                time.sleep(1)\n        except (KeyboardInterrupt,BudgetStop) as exc:\n            interrupted=True;print(str(exc),\'— останавливаю процесс и сохраняю завершённые шаги.\',flush=True)\n            process.terminate()\n            try:process.wait(timeout=min(60,max(1,budget.left(\'hard\')-120)))\n            except subprocess.TimeoutExpired:process.kill();process.wait(timeout=30)\n        finally:\n            if process.poll() is None:\n                process.terminate()\n                try:process.wait(timeout=30)\n                except subprocess.TimeoutExpired:process.kill();process.wait(timeout=30)\n            thread.join(timeout=10)\n        atomic_json(run/\'process_exit.json\',{\'returncode\':process.returncode,\'interrupted\':interrupted,\'remaining_seconds\':budget.left(\'hard\')})\n        return process.returncode\n\ndef archive_results(run):\n    """Full latest optimizer checkpoint + selected export; never pretend a partial snapshot is complete."""\n    from ml.qwen.budget_v21 import verify_snapshot\n    run=Path(run).resolve();keep=set();errors=[];latest_complete=False\n    for label in [\'last\',\'best\']:\n        pointer=run/(label+\'_pointer.json\')\n        if not pointer.is_file():continue\n        try:\n            snap=(run/json.loads(pointer.read_text())[\'relative_path\']).resolve()\n            if not snap.is_relative_to(run/\'checkpoints\'):raise ValueError(\'Unsafe pointer\')\n            verify_snapshot(snap)\n            if label==\'last\':latest_complete=True\n            if label==\'last\' or not (run/\'export/model_manifest.json\').exists():keep.add(snap)\n        except Exception as exc:errors.append({\'pointer\':label,\'error\':str(exc)})\n    archive=run.parent/(run.name+\'_results.zip\');temporary=archive.with_suffix(\'.pending.zip\')\n    selected=[]\n    for p in sorted(run.rglob(\'*\')):\n        if not p.is_file() or p.is_symlink() or \'__pycache__\' in p.parts or p.suffix==\'.pyc\' or \'.pending\' in p.name:continue\n        if (run/\'checkpoints\') in p.parents and not any(s in p.parents for s in keep):continue\n        selected.append(p)\n    def sha(path):\n        h=hashlib.sha256()\n        with path.open(\'rb\') as f:\n            for block in iter(lambda:f.read(4*1024**2),b\'\'):h.update(block)\n        return h.hexdigest()\n    files={str(p.relative_to(run)).replace(\'\\\\\',\'/\'):sha(p) for p in selected}\n    with zipfile.ZipFile(temporary,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as z:\n        for p in selected:z.write(p,str(p.relative_to(run)).replace(\'\\\\\',\'/\'))\n        z.writestr(\'results_archive_manifest.json\',json.dumps({\'files_sha256\':files,\'checkpoint_errors\':errors,\n            \'includes_full_latest_training_state\':latest_complete,\'original_run\':str(run)},ensure_ascii=False,indent=2))\n    with zipfile.ZipFile(temporary) as z:\n        if z.testzip() is not None:raise ValueError(\'Result ZIP CRC failure; original run preserved\')\n    os.replace(temporary,archive)\n    atomic_json(run/\'results_archive.json\',{\'path\':str(archive),\'sha256\':sha(archive),\'bytes\':archive.stat().st_size,\'checkpoint_errors\':errors})\n    print(\'Архив результатов на Google Drive:\',archive,flush=True)\n    return archive\n\ndef show_results(run):\n    print(\'Папка:\',run)\n    report=run/\'experiment_report.json\'\n    if report.is_file():\n        value=json.loads(report.read_text(encoding=\'utf-8\'))\n        print(\'Результат:\',json.dumps({k:value[k] for k in [\'status\',\'new_updates\',\'completed_new_updates\',\'schedule_completed\',\'selected_new_updates\',\'selected_is_original648\',\'stop_reason\',\'reason\'] if k in value},ensure_ascii=False))\n        for label in [\'before\',\'last\',\'selected\']:\n            part=value.get(label)\n            if part:print(label,json.dumps({\'complete\':part.get(\'complete\'),\'metrics\':part.get(\'metrics\'),\'counts\':part.get(\'counts\'),\'failed\':part.get(\'failed_generations\')},ensure_ascii=False))\n        print(\'Полный отчёт:\',report)\n    for name in [\'failure.json\',\'launcher_error.json\',\'training_complete.json\',\'last_pointer.json\',\'best_pointer.json\',\'results_archive.json\']:\n        p=run/name\n        if p.is_file():\n            value=json.loads(p.read_text(encoding=\'utf-8\'));value.pop(\'traceback\',None)\n            print(\'\\n\'+name+\'\\n\'+json.dumps(value,ensure_ascii=False,indent=2))\n    log=run/\'training_log.json\'\n    if log.is_file():\n        entries=json.loads(log.read_text());print(\'\\nПоследние обновления:\',json.dumps(entries[-3:],ensure_ascii=False,indent=2))\n',globals())
RUN,_budget_data=setup_session(_first_cell_started,AVAILABLE_MINUTES,EXISTING_RUN)
ROOT=Path('/content/mts_qwen06_v21_2000')


## 2. Пакет

Выберите **mts_qwen06_v21_2000_payload.zip** из выданной папки, дождитесь загрузки и проверки. Другие архивы не подходят.

In [ ]:
# Загружается один mts_qwen06_v21_2000_payload.zip. Повтор не затирает файлы.
import io,hashlib,json,zipfile,sys
from google.colab import files
exec('"""Notebook upload helper: content identity, repeatable verification, no overwrite."""\nfrom pathlib import Path\nimport hashlib\nimport io\nimport json\nimport zipfile\n\n\ndef prepare_payload(root, uploaded, expected_zip_sha, expected_manifest_sha):\n    root = Path(root).resolve()\n    if not root.exists() or not any(root.iterdir()):\n        matches = [b for b in (uploaded or {}).values()\n                   if hashlib.sha256(b).hexdigest() == expected_zip_sha]\n        if not matches:\n            raise ValueError(\'Среди загруженных файлов нет выданного payload: контрольная сумма отличается\')\n        with zipfile.ZipFile(io.BytesIO(matches[0])) as z:\n            if z.testzip() is not None or sum(i.file_size for i in z.infolist()) >= 256 * 1024**2:\n                raise ValueError(\'Повреждённый или слишком большой payload\')\n            for item in z.infolist():\n                path = (root.parent / item.filename).resolve()\n                if not path.is_relative_to(root) or (item.external_attr >> 16) & 0o170000 == 0o120000:\n                    raise ValueError(\'Недопустимый путь в payload\')\n            z.extractall(root.parent)\n    manifest_path = root/\'manifest.json\'\n    if not manifest_path.is_file() or hashlib.sha256(manifest_path.read_bytes()).hexdigest() != expected_manifest_sha:\n        raise ValueError(\'Manifest отсутствует или изменён. Папка не перезаписана; сохраните результаты и восстановите payload в новой среде\')\n    manifest = json.loads(manifest_path.read_text(encoding=\'utf-8\'))\n    for rel, expected in manifest[\'sha256\'].items():\n        path = (root/rel).resolve()\n        if not path.is_relative_to(root) or not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:\n            raise ValueError(\'Файл отсутствует или изменён, папка не перезаписана: \'+rel)\n    return manifest\n',globals())
uploaded=None
if not ROOT.exists() or not any(ROOT.iterdir()):
    uploaded=files.upload()
manifest=prepare_payload(ROOT,uploaded,'e4213f2a63108222c75715b746b90a7307b5520aff4263825dda2c2cf3a0f2c0','14b481434fd247b640756cf418547db035c12ea7047b673127e45a07f5df4e62')
sys.path.insert(0,str(ROOT))
from ml.qwen.budget_v21 import load_budget
budget=load_budget(RUN/'budget.json')
print('Пакет проверен. Осталось всего, минут:',round(budget.left('hard')/60,1))
del uploaded


## 3. Установка и полная проверка данных

Модель ещё не загружается. При ошибке не переходите к обучению; сохраните полный текст ошибки.

In [ ]:
import subprocess,os
budget=load_budget(RUN/'budget.json')
budget.require(25*60,'train')
subprocess.run([sys.executable,'-m','pip','install','--quiet','-r',str(ROOT/'requirements.txt')],check=True,
    timeout=min(600,max(1,budget.left('train')-15*60)))
# Библиотеки импортируются новым процессом, а не старым состоянием ядра.
subprocess.run([sys.executable,str(ROOT/'check_inputs.py'),'--root',str(ROOT),'--out',str(RUN/'input_checks.json')],
    cwd=ROOT,check=True,timeout=min(600,max(1,budget.left('train')-15*60)),
    env=dict(os.environ,TOKENIZERS_PARALLELISM='false'))
print('Все входы, цели, маски и контекст проверены. Теперь можно запускать ячейку4.')


## 4. Единственный запуск обучения

Сначала исходные ответы32, затем500 или меньше новых обновлений в оставшееся время, первые16 — встроенный пилот. Затем ответы после, выбор, проверка перезагрузки и архив на Drive. Не запускайте ячейку повторно, пока процесс работает.

В конце ВМ автоматически отключится: архив уже будет лежать на Drive. Даже если run неполный/ошибочный, скачайте архив для разбора. Успешное окончание процесса само по себе не означает прохождение метрик хакатона.

In [ ]:
from session_control import supervise,archive_results,show_results,arm_disconnect
budget=load_budget(RUN/'budget.json')
if not RUN_TRAINING:raise RuntimeError('Для реального запуска задайте RUN_TRAINING=True в ячейке1.')
command=[sys.executable,'-u','-m','ml.qwen.v21_train','--root',str(ROOT),'--run',str(RUN),'--budget-file',str(RUN/'budget.json')]
if RESUME_NEW_CHECKPOINT:command+=['--resume-checkpoint',RESUME_NEW_CHECKPOINT]
os.chdir(ROOT)
try:
    returncode=supervise(command,RUN,budget)
    print('Код завершения:',returncode,'(подробности в failure/report, если есть)')
finally:
    # Даже при ошибке или остановке пытаемся забрать уже сохранённые результаты.
    archive=archive_results(RUN)
    show_results(RUN)
    if AUTO_DISCONNECT:
        print('Архив сохранён на Drive. Отключаю ВМ через5 секунд; результат не потеряется.')
        arm_disconnect(time.time()+5)


## 5. Только посмотреть сохранённое

Этот раздел не запускает обучение. Если среда сбросилась, задайте EXISTING_RUN точным ранее напечатанным путём. После просмотра можно отключить ВМ вручную. Архив скачивается через Google Drive, не из временной папки /content.

In [ ]:
# Просмотр после сброса: не нужны переменные RUN/ROOT из предыдущих ячеек.
# При новой ВМ вставьте здесь ранее напечатанный путь папки запуска.
EXISTING_RUN=""
exec('"""Colab orchestration. One persisted wall-clock budget; results survive kernel reset."""\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nimport hashlib,json,os,subprocess,sys,threading,time,zipfile\n\nLOCATOR=Path(\'/content/mts_qwen06_v21_current.json\')\n\ndef atomic_json(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+\'.pending\');tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2)+\'\\n\',encoding=\'utf-8\');os.replace(tmp,path)\n\ndef arm_disconnect(deadline):\n    """VM release is best effort; process supervisor independently enforces its own cutoff."""\n    prior=globals().get(\'_release_timer\')\n    if prior:prior.cancel()\n    def release():\n        try:\n            from google.colab import runtime\n            runtime.unassign()\n        except Exception as exc:print(\'Не удалось отключить ВМ автоматически:\',repr(exc),flush=True)\n    timer=threading.Timer(max(0,deadline-time.time()),release);timer.daemon=True;timer.start()\n    globals()[\'_release_timer\']=timer\n    return timer\n\ndef setup_session(origin,available_minutes=300,existing_run=\'\',*,mount=True,arm=True):\n    if not 60<=available_minutes<=300:raise ValueError(\'AVAILABLE_MINUTES: от60 до300, с учётом уже использованного сегодня времени\')\n    if arm:arm_disconnect(origin+available_minutes*60)\n    if mount:\n        from google.colab import drive\n        drive.mount(\'/content/drive\')\n    if existing_run:\n        run=Path(existing_run)\n        if not (run/\'budget.json\').is_file():raise ValueError(\'EXISTING_RUN не содержит budget.json; не создаём пустую замену\')\n    elif LOCATOR.exists():\n        run=Path(json.loads(LOCATOR.read_text())[\'run\'])\n        if not (run/\'budget.json\').is_file():raise ValueError(\'Предыдущая папка недоступна; проверьте Drive/EXISTING_RUN\')\n    else:\n        run=Path(\'/content/drive/MyDrive/MTS_Qwen06_V21_runs\')/datetime.fromtimestamp(origin,timezone.utc).strftime(\'v21_%Y%m%dT%H%M%SZ\')\n        run.mkdir(parents=True,exist_ok=False)\n        atomic_json(run/\'budget.json\',{\'origin_epoch\':origin,\'total_minutes\':available_minutes,\'scope\':\'Entire experiment, not a guaranteed Google quota\'})\n    budget=json.loads((run/\'budget.json\').read_text());deadline=budget[\'origin_epoch\']+budget[\'total_minutes\']*60\n    if available_minutes!=budget[\'total_minutes\']:raise ValueError(\'Повторный запуск не меняет бюджет; сохранённый размер: \'+str(budget[\'total_minutes\']))\n    atomic_json(LOCATOR,{\'run\':str(run),\'root\':\'/content/mts_qwen06_v21_2000\'})\n    if arm:arm_disconnect(deadline)\n    print(\'Папка RUN:\',run,\'\\nОсталось общего времени, минут:\',round(max(0,deadline-time.time())/60,1),flush=True)\n    print(\'Бюджет включает подготовку, загрузку, обучение, оценку и архив. Доступность GPU Google не гарантируется.\')\n    return run,budget\n\ndef recover_run(existing_run=\'\'):\n    if existing_run:run=Path(existing_run)\n    elif LOCATOR.exists():run=Path(json.loads(LOCATOR.read_text())[\'run\'])\n    else:\n        candidates=sorted(Path(\'/content/drive/MyDrive/MTS_Qwen06_V21_runs\').glob(\'v21_*\'))\n        raise ValueError(\'Укажите EXISTING_RUN явно, чтобы не выбрать чужой запуск: \'+str([str(p) for p in candidates]))\n    if not (run/\'budget.json\').is_file():raise ValueError(\'Нет сохранённого budget.json: \'+str(run))\n    return run\n\ndef supervise(command,run,budget):\n    from ml.qwen.budget_v21 import BudgetStop\n    budget.require(15*60)\n    if (run/\'experiment_report.json\').is_file():\n        report=json.loads((run/\'experiment_report.json\').read_text())\n        if report.get(\'status\')==\'completed\' or \'--resume-checkpoint\' not in command:\n            raise ValueError(\'В этой папке уже есть отчёт. Используйте просмотр результатов; незавершённый запуск возобновляется явно из своего чекпоинта.\')\n    env=dict(os.environ,TOKENIZERS_PARALLELISM=\'false\',HF_HUB_DISABLE_PROGRESS_BARS=\'1\',PYTHONUNBUFFERED=\'1\')\n    with (run/\'console.log\').open(\'a\',encoding=\'utf-8\',buffering=1) as log:\n        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding=\'utf-8\',errors=\'replace\',bufsize=1,env=env)\n        def reader():\n            for line in process.stdout:\n                log.write(line);print(line,end=\'\',flush=True)\n        thread=threading.Thread(target=reader,daemon=True);thread.start()\n        interrupted=False\n        try:\n            while process.poll() is None:\n                if budget.left(\'supervisor\')<=0:raise BudgetStop(\'Пора сохранить результаты и освободить GPU\')\n                time.sleep(1)\n        except (KeyboardInterrupt,BudgetStop) as exc:\n            interrupted=True;print(str(exc),\'— останавливаю процесс и сохраняю завершённые шаги.\',flush=True)\n            process.terminate()\n            try:process.wait(timeout=min(60,max(1,budget.left(\'hard\')-120)))\n            except subprocess.TimeoutExpired:process.kill();process.wait(timeout=30)\n        finally:\n            if process.poll() is None:\n                process.terminate()\n                try:process.wait(timeout=30)\n                except subprocess.TimeoutExpired:process.kill();process.wait(timeout=30)\n            thread.join(timeout=10)\n        atomic_json(run/\'process_exit.json\',{\'returncode\':process.returncode,\'interrupted\':interrupted,\'remaining_seconds\':budget.left(\'hard\')})\n        return process.returncode\n\ndef archive_results(run):\n    """Full latest optimizer checkpoint + selected export; never pretend a partial snapshot is complete."""\n    from ml.qwen.budget_v21 import verify_snapshot\n    run=Path(run).resolve();keep=set();errors=[];latest_complete=False\n    for label in [\'last\',\'best\']:\n        pointer=run/(label+\'_pointer.json\')\n        if not pointer.is_file():continue\n        try:\n            snap=(run/json.loads(pointer.read_text())[\'relative_path\']).resolve()\n            if not snap.is_relative_to(run/\'checkpoints\'):raise ValueError(\'Unsafe pointer\')\n            verify_snapshot(snap)\n            if label==\'last\':latest_complete=True\n            if label==\'last\' or not (run/\'export/model_manifest.json\').exists():keep.add(snap)\n        except Exception as exc:errors.append({\'pointer\':label,\'error\':str(exc)})\n    archive=run.parent/(run.name+\'_results.zip\');temporary=archive.with_suffix(\'.pending.zip\')\n    selected=[]\n    for p in sorted(run.rglob(\'*\')):\n        if not p.is_file() or p.is_symlink() or \'__pycache__\' in p.parts or p.suffix==\'.pyc\' or \'.pending\' in p.name:continue\n        if (run/\'checkpoints\') in p.parents and not any(s in p.parents for s in keep):continue\n        selected.append(p)\n    def sha(path):\n        h=hashlib.sha256()\n        with path.open(\'rb\') as f:\n            for block in iter(lambda:f.read(4*1024**2),b\'\'):h.update(block)\n        return h.hexdigest()\n    files={str(p.relative_to(run)).replace(\'\\\\\',\'/\'):sha(p) for p in selected}\n    with zipfile.ZipFile(temporary,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as z:\n        for p in selected:z.write(p,str(p.relative_to(run)).replace(\'\\\\\',\'/\'))\n        z.writestr(\'results_archive_manifest.json\',json.dumps({\'files_sha256\':files,\'checkpoint_errors\':errors,\n            \'includes_full_latest_training_state\':latest_complete,\'original_run\':str(run)},ensure_ascii=False,indent=2))\n    with zipfile.ZipFile(temporary) as z:\n        if z.testzip() is not None:raise ValueError(\'Result ZIP CRC failure; original run preserved\')\n    os.replace(temporary,archive)\n    atomic_json(run/\'results_archive.json\',{\'path\':str(archive),\'sha256\':sha(archive),\'bytes\':archive.stat().st_size,\'checkpoint_errors\':errors})\n    print(\'Архив результатов на Google Drive:\',archive,flush=True)\n    return archive\n\ndef show_results(run):\n    print(\'Папка:\',run)\n    report=run/\'experiment_report.json\'\n    if report.is_file():\n        value=json.loads(report.read_text(encoding=\'utf-8\'))\n        print(\'Результат:\',json.dumps({k:value[k] for k in [\'status\',\'new_updates\',\'completed_new_updates\',\'schedule_completed\',\'selected_new_updates\',\'selected_is_original648\',\'stop_reason\',\'reason\'] if k in value},ensure_ascii=False))\n        for label in [\'before\',\'last\',\'selected\']:\n            part=value.get(label)\n            if part:print(label,json.dumps({\'complete\':part.get(\'complete\'),\'metrics\':part.get(\'metrics\'),\'counts\':part.get(\'counts\'),\'failed\':part.get(\'failed_generations\')},ensure_ascii=False))\n        print(\'Полный отчёт:\',report)\n    for name in [\'failure.json\',\'launcher_error.json\',\'training_complete.json\',\'last_pointer.json\',\'best_pointer.json\',\'results_archive.json\']:\n        p=run/name\n        if p.is_file():\n            value=json.loads(p.read_text(encoding=\'utf-8\'));value.pop(\'traceback\',None)\n            print(\'\\n\'+name+\'\\n\'+json.dumps(value,ensure_ascii=False,indent=2))\n    log=run/\'training_log.json\'\n    if log.is_file():\n        entries=json.loads(log.read_text());print(\'\\nПоследние обновления:\',json.dumps(entries[-3:],ensure_ascii=False,indent=2))\n',globals())
from google.colab import drive
drive.mount('/content/drive')
RUN=recover_run(EXISTING_RUN)
show_results(RUN)
print('Архив *_results.zip находится рядом с папкой RUN. Скачайте его через Google Drive.')
